## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## 2. Load Dataset

In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data"

iris_df = pd.read_csv(url, header=None)

print("Dataset Shape:", iris_df.shape)
iris_df.head()


## 3. Prepare Data

In [ ]:
target_labels = iris_df.iloc[:100, 4].values

target_labels = np.where(
    target_labels == "Iris-setosa",
    0,
    1
)

feature_matrix = iris_df.iloc[:100, [0, 2]].values

print("Features Shape:", feature_matrix.shape)
print("Labels Shape:", target_labels.shape)


## 4. Create Train-Test Split From Scratch

In [ ]:
np.random.seed(42)

all_indices = np.random.permutation(
    len(feature_matrix)
)

test_size = int(
    0.20 * len(feature_matrix)
)

test_indices = all_indices[:test_size]
train_indices = all_indices[test_size:]

X_train = feature_matrix[train_indices]
X_test = feature_matrix[test_indices]

y_train = target_labels[train_indices]
y_test = target_labels[test_indices]

print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))


## 5. Visualize Training Data

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    X_train[y_train == 0, 0],
    X_train[y_train == 0, 1],
    label="Setosa"
)

plt.scatter(
    X_train[y_train == 1, 0],
    X_train[y_train == 1, 1],
    label="Versicolor"
)

plt.xlabel("Sepal Length")
plt.ylabel("Petal Length")
plt.title("Training Data Distribution")
plt.legend()

plt.show()


## 6. Logistic Regression From Scratch

In [ ]:
class LogisticRegressionClassifier:

    def __init__(self, learning_rate=0.1, n_epochs=1000, standardize=True):
        self.learning_rate = learning_rate
        self.n_epochs = n_epochs
        self.standardize = standardize

        self.weights = None
        self.bias = None
        self.loss_history = []

        self.feature_mean_ = None
        self.feature_std_ = None

    def _scale(self, X):
        if not self.standardize:
            return X
        return (X - self.feature_mean_) / self.feature_std_

    @staticmethod
    def _sigmoid(z):
        z = np.clip(z, -500, 500)
        return 1.0 / (1.0 + np.exp(-z))

    def train(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)

        n_samples, n_features = X.shape

        if self.standardize:
            self.feature_mean_ = X.mean(axis=0)
            self.feature_std_ = X.std(axis=0)
            self.feature_std_[self.feature_std_ == 0] = 1.0  

        X_scaled = self._scale(X)

        self.weights = np.zeros(n_features)
        self.bias = 0.0
        self.loss_history = []

        epsilon = 1e-15  # avoids log(0)

        for _ in range(self.n_epochs):
            linear_output = X_scaled @ self.weights + self.bias
            y_pred = self._sigmoid(linear_output)

            loss = -np.mean(
                y * np.log(y_pred + epsilon) +
                (1 - y) * np.log(1 - y_pred + epsilon)
            )
            self.loss_history.append(loss)

            # Gradients
            error = y_pred - y
            grad_weights = (X_scaled.T @ error) / n_samples
            grad_bias = np.mean(error)

            # Parameter update
            self.weights -= self.learning_rate * grad_weights
            self.bias -= self.learning_rate * grad_bias

        return self

    def predict_proba(self, X):
        X = np.asarray(X, dtype=float)
        X_scaled = self._scale(X)
        linear_output = X_scaled @ self.weights + self.bias
        return self._sigmoid(linear_output)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

    def calculate_accuracy(self, X, y):
        y = np.asarray(y)
        predictions = self.predict(X)
        return np.mean(predictions == y)


## 7. Train Model

In [ ]:
model = LogisticRegressionClassifier(
    learning_rate=0.1,
    n_epochs=1000
)

model.train(
    X_train,
    y_train
)


## 8. Plot Loss Curve

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    model.loss_history
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss")

plt.show()


## 9. Evaluate Model

In [ ]:
train_accuracy = model.calculate_accuracy(
    X_train,
    y_train
)

test_accuracy = model.calculate_accuracy(
    X_test,
    y_test
)

print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Testing Accuracy : {test_accuracy:.4f}")


## 11. Decision Boundary Visualization

In [ ]:
x_min = X_train[:, 0].min() - 1
x_max = X_train[:, 0].max() + 1

y_min = X_train[:, 1].min() - 1
y_max = X_train[:, 1].max() + 1

xx, yy = np.meshgrid(
    np.arange(x_min, x_max, 0.02),
    np.arange(y_min, y_max, 0.02)
)

grid_points = np.c_[xx.ravel(), yy.ravel()]

predictions = model.predict(grid_points)

predictions = predictions.reshape(xx.shape)

plt.figure(figsize=(8, 6))

plt.contourf(
    xx,
    yy,
    predictions,
    alpha=0.3
)

plt.scatter(
    X_train[:, 0],
    X_train[:, 1],
    c=y_train
)

plt.xlabel("Sepal Length")
plt.ylabel("Petal Length")
plt.title("Decision Boundary")

plt.show()
